# Soft-Mixture head-first DQA-MoX

- created_utc: 2026-05-11T11:32:49+00:00
- target_mAP50: 0.600
- workspace: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27a_soft_mixture_head_first_40_10`
- log: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27a_soft_mixture_head_first_40_10/logs/27a_soft_mixture_head_first_40_10_train.log`
- research_note: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/reports/27_research_note_iter_000_27a_soft_mixture_head_first_40_10.md`

## Current Results

| trial | best mAP50 | mAP50:95 |
|---|---:|---:|
| 24a_client_dominant_soft_expand | 0.457000 | 0.258000 |

## Hypothesis

25aのbackbone-firstが途中mAPを押し上げ切れていないので、FedMoXのSoft-Mixture思想を優先し、長いneck/head専門化でpseudoGTをhead側の専門家に吸わせてから短くfull更新する。

## Paper Basis

- FedMoX/PSSFL: https://arxiv.org/abs/2508.16568
  FedMoX treats the practical setting as server labeled high-resolution data plus client unlabeled low-resolution data, and uses sparse MoE with a spatial router and Soft-Mixture to stabilize semi-supervised FL.
- FedSTO: https://arxiv.org/abs/2310.17097
  FedSTO uses server-only labels and client-only unlabeled non-IID data; its two-stage training selectively refines detector parts first, then moves to full-parameter training.
- PseCo: https://arxiv.org/abs/2203.16317
  PseCo argues that classification score alone does not guarantee localization precision; prediction-guided assignment and consistency voting make learning robust to coarse boxes.


In [ ]:
import csv
import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path('/app/Object_Detection')
WORKSPACE = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27a_soft_mixture_head_first_40_10')
LOG_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27a_soft_mixture_head_first_40_10/logs/27a_soft_mixture_head_first_40_10_train.log')
METRICS_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27a_soft_mixture_head_first_40_10/stats/18_client_balanced_single_injection_dqamox_final_metrics.csv')
CMD = ['/opt/venv/bin/python', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/scripts/run_scene_daynight_dqa_18_client_balanced_single_injection_dqamox.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27a_soft_mixture_head_first_40_10', '--repair-baseline-rounds', '0', '--source-workspace', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/08_full_latent_dqamox_from_warmup', '--source-repair-baseline-rounds', '30', '--target-map50', '0.6', '--num-experts', '4', '--top-k', '2', '--router-temperature', '1.25', '--router-balance-weight', '0.02', '--router-entropy-weight', '0.0005', '--dqa-client-balance-stats', '--dqa-client-balance-target', 'median', '--dqa-client-balance-max-scale', '4.0', '--load-bias-strength', '0.25', '--batch-size', '80', '--workers', '8', '--gpus', '2', '--max-images-per-client', '0', '--master-port', '39000', '--evaluate', '--classwise', '--no-eval-plots', '--force', '--warmup-epochs', '50', '--client-limit', '3000', '--client-sampling-ratio', '0.333', '--client-sampling-seed', '270001', '--phase1-rounds', '40', '--phase2-rounds', '10', '--phase1-train-scope', 'neck_head', '--phase1-repair-train-scope', 'neck_head', '--phase1-client-epochs', '1', '--phase1-client-lr', '0.00034', '--phase1-source-repeat', '1', '--phase1-pseudo-repeat', '3', '--phase1-loss-box', '0.0007', '--phase2-train-scope', 'all', '--phase2-repair-train-scope', 'all', '--phase2-client-epochs', '1', '--phase2-client-lr', '0.000045', '--phase2-source-repeat', '1', '--phase2-pseudo-repeat', '1', '--phase2-loss-box', '0.00008', '--server-repair-epochs', '1', '--server-repair-lr', '0.00020', '--server-repair-loss-box', '0.012', '--dqa-server-anchor', '0.18', '--dqa-min-server-alpha', '0.12', '--dqa-residual-blend', '0.00', '--late-dqa-server-anchor', '0.09', '--late-dqa-min-server-alpha', '0.04', '--late-dqa-residual-blend', '0.00', '--curriculum-start-round', '41', '--expert-keep-fraction', '0.88', '--expert-max-class-fraction', '0.36', '--actual-max-class-fraction', '0.52', '--late-expert-keep-fraction', '0.95', '--late-expert-max-class-fraction', '0.42', '--late-actual-max-class-fraction', '0.60', '--min-score', '0.17', '--min-stability', '0.52', '--late-min-score', '0.12', '--late-min-stability', '0.42', '--max-boxes-per-image', '14']

WORKSPACE.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
(WORKSPACE / "stats").mkdir(parents=True, exist_ok=True)
(WORKSPACE / "stats" / "27_notebook_command.json").write_text(
    json.dumps({"command": CMD}, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print(" ".join(CMD))
with LOG_PATH.open("w", encoding="utf-8") as log:
    proc = subprocess.run(CMD, cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT, check=False)
print("returncode", proc.returncode)
print("log", LOG_PATH)
if proc.returncode != 0:
    raise SystemExit(proc.returncode)


In [ ]:
import csv
from pathlib import Path

METRICS_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27a_soft_mixture_head_first_40_10/stats/18_client_balanced_single_injection_dqamox_final_metrics.csv')
rows = list(csv.DictReader(METRICS_PATH.open(encoding="utf-8"))) if METRICS_PATH.exists() else []
for row in rows:
    print(row)
